# Submission 2: Machine Learning Pipeline & Sistem MLOps — Prediksi Risiko Diabetes
**Nama:** yusriiii
**Proyek:** Pengembangan dan Pengoperasian Sistem Machine Learning (TFX + Apache Beam + Docker + Prometheus)

Notebook ini mendokumentasikan seluruh tahapan pembangunan *machine learning pipeline* menggunakan **TensorFlow Extended (TFX)**, yang dijalankan menggunakan **Apache Beam** sebagai *pipeline orchestrator* (bukan `InteractiveContext`). Seluruh definisi komponen pipeline dipisahkan ke dalam modul-modul terstruktur pada folder `modules/` (prinsip *clean code*): `transform.py`, `tuner.py`, `trainer.py`, dan `components.py`.

Pipeline ini juga menerapkan komponen **Tuner** untuk melakukan pencarian hyperparameter (hyperparameter tuning) secara otomatis sebelum model final dilatih oleh komponen Trainer.

## Daftar Isi
1. Import & Konfigurasi Path
2. Ringkasan Dataset
3. Penjelasan Komponen Pipeline (ExampleGen s.d. Pusher)
4. Menjalankan Pipeline dengan Apache Beam (BeamDagRunner)
5. Hasil Tuning, Training, dan Evaluasi Model
6. Kesimpulan


In [1]:
import os
import sys
import json
import pandas as pd
import tensorflow as tf
import tfx

print("TensorFlow version:", tf.__version__)
print("TFX version:", tfx.__version__)

BASE_DIR = os.getcwd()
MODULE_ROOT = os.path.join(BASE_DIR, "modules")
sys.path.insert(0, MODULE_ROOT)


I0000 00:00:1784804197.076603     551 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784804197.077449     551 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1784804197.246612     551 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1784804199.309308     551 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784804199.309853     551 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TensorFlow version: 2.21.0
TFX version: 1.21.0


## 1. Ringkasan Dataset

Dataset yang digunakan pada proyek ini adalah **Pima Indians Diabetes Dataset**, dataset klasik untuk kasus klasifikasi biner risiko diabetes, terdiri dari **768 baris data** pasien wanita keturunan Pima Indian berusia minimal 21 tahun, dengan **8 fitur klinis numerik** dan **1 label target biner** (`Outcome`: 1 = menderita diabetes, 0 = tidak).


In [2]:
df = pd.read_csv("data/diabetes.csv")
print("Jumlah baris & kolom:", df.shape)
print(df["Outcome"].value_counts())
df.head()


Jumlah baris & kolom: (768, 9)
Outcome
0    500
1    268
Name: count, dtype: int64


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## 2. Penjelasan Komponen Pipeline

Seluruh komponen pipeline didefinisikan secara modular pada berkas `modules/components.py` (lihat fungsi `create_components`). Berikut penjelasan tiap komponen:

1. **ExampleGen (`CsvExampleGen`)** — membaca dataset CSV dan membaginya menjadi data *train* (80%) dan *eval* (20%), lalu mengonversinya ke format `tf.Example`.
2. **StatisticsGen** — menghasilkan statistik deskriptif dari data (mean, std, missing value, dsb).
3. **SchemaGen** — membuat *schema* data secara otomatis dari statistik yang dihasilkan.
4. **ExampleValidator** — memvalidasi data terhadap schema untuk mendeteksi anomali.
5. **Transform** (`modules/transform.py`) — melakukan normalisasi z-score pada seluruh 8 fitur numerik.
6. **Tuner** (`modules/tuner.py`) — menjalankan pencarian hyperparameter otomatis (KerasTuner `RandomSearch`, 8 trial) untuk menemukan kombinasi `units_1`, `units_2`, `dropout_rate`, dan `learning_rate` terbaik berdasarkan `val_binary_accuracy`.
7. **Trainer** (`modules/trainer.py`) — melatih model final Neural Network menggunakan hyperparameter terbaik hasil Tuner.
8. **Resolver** (`LatestBlessedModelStrategy`) — mengambil model *blessed* terakhir sebagai baseline pembanding.
9. **Evaluator** — mengevaluasi model menggunakan TensorFlow Model Analysis (TFMA) dengan threshold kelulusan `binary_accuracy >= 0.6`.
10. **Pusher** — menyimpan model ke direktori `serving_model` hanya jika model dinyatakan *blessed*.

Detail arsitektur model, hyperparameter yang di-tuning, dan konfigurasi masing-masing komponen dapat dilihat langsung pada berkas-berkas di folder `modules/`.


In [3]:
with open("modules/components.py") as f:
    print(f.read()[:1500])
print("... (lihat berkas modules/components.py untuk kode lengkap) ...")


"""components.py

Modul ini bertanggung jawab untuk membangun (assemble) seluruh komponen
TFX yang dibutuhkan pada pipeline prediksi risiko diabetes. Dipisah dari
notebook/skrip utama agar pipeline definition mudah dibaca, diuji, dan
digunakan ulang (clean code / separation of concerns).
"""

from tfx.components import (
    CsvExampleGen,
    StatisticsGen,
    SchemaGen,
    ExampleValidator,
    Transform,
    Tuner,
    Trainer,
    Evaluator,
    Pusher,
)
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import (
    LatestBlessedModelStrategy,
)
from tfx.proto import example_gen_pb2, trainer_pb2, pusher_pb2
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing
import tensorflow_model_analysis as tfma


def create_example_gen(data_root: str) -> CsvExampleGen:
    """Membuat komponen ExampleGen yang membaca data CSV dan membaginya
    menjadi split train (80%) dan eva

## 3. Menjalankan Pipeline dengan Apache Beam (BeamDagRunner)

Berbeda dengan submission sebelumnya yang menggunakan `InteractiveContext`, pipeline pada proyek ini dirangkai menjadi satu objek `tfx.orchestration.pipeline.Pipeline` dan dijalankan sekaligus (end-to-end) menggunakan **`BeamDagRunner`**, yaitu runner yang mengeksekusi pipeline TFX menggunakan **Apache Beam** sebagai *orchestrator*. Seluruh artefak pipeline disimpan pada folder **`yusriiii-pipeline/`**.

Definisi pipeline (`init_local_pipeline`) dan proses eksekusinya berada pada berkas `pipeline.py`, dan dipanggil di bawah ini.


In [4]:
sys.path.insert(0, BASE_DIR)
from pipeline import run_pipeline, PIPELINE_ROOT, METADATA_PATH  # noqa: E402

print("Pipeline root  :", PIPELINE_ROOT)
print("Metadata path  :", METADATA_PATH)

run_pipeline()


INFO:absl:Finished tuning... Tuner ID: tuner0


INFO:absl:Best HyperParameters: {'space': [{'class_name': 'Int', 'config': {'name': 'units_1', 'default': None, 'conditions': [], 'min_value': 16, 'max_value': 128, 'step': 16, 'sampling': 'linear'}}, {'class_name': 'Int', 'config': {'name': 'units_2', 'default': None, 'conditions': [], 'min_value': 8, 'max_value': 64, 'step': 8, 'sampling': 'linear'}}, {'class_name': 'Float', 'config': {'name': 'dropout_rate', 'default': 0.1, 'conditions': [], 'min_value': 0.1, 'max_value': 0.5, 'step': 0.1, 'sampling': 'linear'}}, {'class_name': 'Choice', 'config': {'name': 'learning_rate', 'default': 0.01, 'conditions': [], 'values': [0.01, 0.001, 0.0001], 'ordered': True}}], 'values': {'units_1': 80, 'units_2': 24, 'dropout_rate': 0.30000000000000004, 'learning_rate': 0.01}}


INFO:absl:Best Hyperparameters are written to /home/claude/project2/yusriiii-pipeline/Tuner/best_hyperparameters/7/best_hyperparameters.txt.


INFO:absl:Tuner results are written to /home/claude/project2/yusriiii-pipeline/Tuner/tuner_results/7/tuner_results.json.


INFO:absl:Cleaning up stateless execution info.


...

INFO:absl:Cleaning up stateful execution info.


INFO:absl:Deleted stateful_working_dir /home/claude/project2/yusriiii-pipeline/Pusher/.system/stateful_working_dir/acc73271-f9c7-4630-8217-4afb50304d4d


INFO:absl:Publishing output artifacts defaultdict(<class 'list'>, {'pushed_model': [Artifact(artifact: uri: "/home/claude/project2/yusriiii-pipeline/Pusher/pushed_model/10"
, artifact_type: name: "PushedModel"
base_type: MODEL
)]}) for execution 10


INFO:absl:MetadataStore with DB connection initialized


INFO:absl:node Pusher is finished.


## 4. Hasil Hyperparameter Tuning

Berikut adalah kombinasi hyperparameter terbaik yang ditemukan oleh komponen **Tuner** (KerasTuner `RandomSearch`, 8 trial, objective `val_binary_accuracy`), yang kemudian digunakan oleh komponen **Trainer** untuk melatih model final.


In [5]:
import glob

best_hp_dir = sorted(glob.glob(os.path.join(PIPELINE_ROOT, "Tuner", "best_hyperparameters", "*")))[-1]
best_hp_file = os.path.join(best_hp_dir, "best_hyperparameters.txt")

with open(best_hp_file) as f:
    best_hp = json.load(f)

print("Hyperparameter terbaik hasil Tuner:")
print(json.dumps(best_hp["values"], indent=2))


Hyperparameter terbaik hasil Tuner:
{
  "units_1": 80,
  "units_2": 24,
  "dropout_rate": 0.30000000000000004,
  "learning_rate": 0.01
}


## 5. Hasil Evaluasi Model

Berikut adalah metrik hasil evaluasi model final (setelah dilatih menggunakan hyperparameter terbaik) pada data eval, menggunakan TensorFlow Model Analysis (TFMA).


In [6]:
import tensorflow_model_analysis as tfma

eval_dir = sorted(glob.glob(os.path.join(PIPELINE_ROOT, "Evaluator", "evaluation", "*")))[-1]
eval_result = tfma.load_eval_result(eval_dir)

for slicing_key, metrics in eval_result.slicing_metrics:
    print("Slice:", slicing_key if slicing_key else "(overall)")
    for metric_name, metric_value in metrics[""][""].items():
        value = list(metric_value.values())[0]
        print(f"  {metric_name}: {value}")


Slice: (overall)
  example_count: 182.0
  auc: 0.8526147959183673
  precision: 0.7884615384615384
  recall: 0.5857142857142857
  binary_accuracy: 0.7802197802197802


In [7]:
blessing_dir = sorted(glob.glob(os.path.join(PIPELINE_ROOT, "Evaluator", "blessing", "*")))[-1]
print("Isi direktori blessing:", os.listdir(blessing_dir))
print("Model dinyatakan BLESSED" if "BLESSED" in os.listdir(blessing_dir) else "Model TIDAK lolos validasi (NOT_BLESSED)")

pushed_model_dir = sorted(glob.glob(os.path.join(PIPELINE_ROOT, "serving_model", "*")))[-1]
print("Model final tersimpan di:", pushed_model_dir)


Isi direktori blessing: ['BLESSED']
Model dinyatakan BLESSED
Model final tersimpan di: /home/claude/project2/yusriiii-pipeline/serving_model/1784804401


## 6. Kesimpulan

Pipeline machine learning untuk prediksi risiko diabetes telah berhasil dijalankan end-to-end menggunakan **Apache Beam** sebagai pipeline orchestrator, melalui seluruh 9 komponen wajib (ExampleGen, StatisticsGen, SchemaGen, ExampleValidator, Transform, Trainer, Resolver, Evaluator, Pusher) ditambah komponen **Tuner** untuk hyperparameter tuning otomatis. Seluruh artefak tersimpan pada folder `yusriiii-pipeline/`.

Model yang dihasilkan dinyatakan **BLESSED** dan otomatis disimpan oleh Pusher ke direktori `serving_model/`, sehingga siap untuk tahap selanjutnya: **containerization dengan Docker**, **deployment ke cloud (Railway)**, dan **monitoring menggunakan Prometheus** — dijelaskan lebih lanjut pada berkas `README.md`.
